In [ ]:
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
import torch
from transformers import BertForSequenceClassification
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from torch import nn

df = pd.read_csv("/content/MovieReviews_NoNoise.csv")

df = df[["Reviews", "emotion"]]

remove_classes = ["surprise", "disgust", "fear", "anger", "anticipation", "optimism"]
df = df[~df["emotion"].isin(remove_classes)]

sadness_df = df[df["emotion"] == "sadness"].sample(frac=0.5, random_state=42)
other_df   = df[df["emotion"] != "sadness"]

df = pd.concat([sadness_df, other_df]).sample(frac=1, random_state=42).reset_index(drop=True)

df["word_count"] = df["Reviews"].astype(str).apply(lambda x: len(x.split()))
df = df[df["word_count"] <= 394]  # 394 kelime * 1.3 ≈ 512 token
df = df.drop(columns=["word_count"])

print(df["emotion"].value_counts())
print("Toplam örnek:", len(df))


df = df.dropna()

le = LabelEncoder()
df["label_id"] = le.fit_transform(df["emotion"])

print(le.classes_)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["Reviews"].tolist(),
    df["label_id"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"]
)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
val_encodings   = tokenizer(val_texts,   truncation=True, padding=True, max_length=512)


class MovieDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = MovieDataset(train_encodings, train_labels)
val_dataset = MovieDataset(val_encodings, val_labels)

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(le.classes_)
)

def compute_metrics(pred):
    logits, labels = pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro"
    )

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    logging_steps=100,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

results = trainer.evaluate()

logs = pd.DataFrame(trainer.state.log_history)

eval_logs = logs[logs["eval_loss"].notnull()].sort_values("epoch")
epochs = eval_logs["epoch"].values


save_path = "./movie_sentiment_bert"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
joblib.dump(le, f"{save_path}/label_encoder.pkl")

print("Model saved to:", save_path)


In [ ]:
import matplotlib.pyplot as plt

eval_logs = logs[logs["eval_loss"].notnull()].copy()

epochs = eval_logs["epoch"]

# --- Metrics plot ---
plt.figure(figsize=(12,6))

print(eval_logs["eval_accuracy"])
plt.plot(epochs, eval_logs["eval_accuracy"], marker='o', label="Accuracy")
plt.plot(epochs, eval_logs["eval_f1"], marker='o', label="F1 Score")
plt.plot(epochs, eval_logs["eval_precision"], marker='o', label="Precision")
plt.plot(epochs, eval_logs["eval_recall"], marker='o', label="Recall")

plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Evaluation Metrics per Epoch")
plt.legend()
plt.grid(True)
plt.show()


# --- Loss plot (separate) ---
plt.figure(figsize=(12,4))

plt.plot(epochs, eval_logs["eval_loss"], marker='o', color="red", label="Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Evaluation Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()